# Modelado del Número de Puntos Anotados en Partidos de la NBA como un Proceso de Poisson

**Materia:** Procesos Estocásticos  
**Proyecto Final**

---

## 1. Introducción

En el baloncesto profesional de la NBA, cada partido es una realización de un fenómeno aleatorio complejo. Sin embargo, la cantidad de puntos anotados por un equipo en un partido puede aproximarse como el conteo de "eventos" discretos (canastas, tiros libres) que ocurren durante un intervalo fijo de tiempo (48 minutos de juego).

Este proyecto investiga si la variable aleatoria *"puntos anotados por partido"* puede modelarse mediante una **distribución de Poisson**, y por extensión, si el proceso subyacente de anotación puede considerarse un **proceso de Poisson**.

## 2. Marco Teórico

### 2.1 Variable Aleatoria Discreta
Una variable aleatoria $X$ es **discreta** si toma un número finito o infinito numerable de valores. Los puntos anotados por partido son una variable discreta: $X \in \{0, 1, 2, \dots\}$.

### 2.2 Distribución de Poisson
Una variable aleatoria discreta $X$ sigue una distribución de Poisson con parámetro $\lambda > 0$, denotada $X \sim \text{Poisson}(\lambda)$, si su función de masa de probabilidad es:

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}, \quad k = 0, 1, 2, \dots$$

### 2.3 Propiedades Fundamentales
- **Esperanza:** $E[X] = \lambda$
- **Varianza:** $\text{Var}(X) = \lambda$
- **Propiedad clave:** $E[X] = \text{Var}(X) = \lambda$

### 2.4 Proceso de Poisson
Un proceso de Poisson es un proceso de conteo $\{N(t), t \geq 0\}$ que satisface:
1. $N(0) = 0$
2. Incrementos independientes
3. Incrementos estacionarios
4. $P(N(h) = 1) = \lambda h + o(h)$
5. $P(N(h) \geq 2) = o(h)$

### 2.5 Hipótesis
**H₀:** Los puntos anotados por partido siguen una distribución de Poisson.  
**H₁:** Los puntos anotados por partido NO siguen una distribución de Poisson.

## 3. Tecnologías Utilizadas
- `nba_api` — Extracción de datos oficiales de NBA.com
- `pandas` — Manipulación y limpieza de datos
- `numpy` — Cálculos numéricos y simulación
- `matplotlib` — Visualización de datos
- `scipy.stats` — Pruebas estadísticas (chi-cuadrada, KS)

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy import stats
from scipy.stats import poisson

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

PROJECT_ROOT = os.path.dirname(os.path.abspath('.'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.extract import extract, TEAM_IDS
from src.clean import clean
from src.analysis import (
    estadistica_descriptiva, verificar_propiedad_poisson,
    poisson_pmf_teorica, frecuencias_por_bins,
    prueba_chi_cuadrada, prueba_ks, simular_poisson,
    graficar_histograma_comparativo, graficar_qq_poisson,
    graficar_serie_temporal, graficar_barras_chi2,
    graficar_media_varianza_por_temporada,
    resumen_estadistico, imprimir_resumen
)

print('Todas las librerías cargadas correctamente.')
print(f'nba_api disponible: {True}')

---
## 4. Selección del Equipo y Extracción de Datos

Seleccionamos un equipo de la NBA para analizar. Por defecto usamos **Golden State Warriors** (team_id=1610612744), uno de los equipos más ofensivos de la última década. Los datos se obtienen de la API oficial de NBA.com usando la librería `nba_api`.

In [ ]:
EQUIPO = 'warriors'
NOMBRE_EQUIPO = 'Golden State Warriors'
SEASONS = [f'{y}-{str(y+1)[-2:]}' for y in range(2014, 2025)]

print(f'Equipo seleccionado: {NOMBRE_EQUIPO}')
print(f'Temporadas a analizar: {len(SEASONS)} ({SEASONS[0]} a {SEASONS[-1]})')

df_raw, raw_path = extract(
    team_name=EQUIPO,
    seasons=SEASONS,
    output_dir=os.path.join('data', 'raw')
)

print(f'\nShape del dataset crudo: {df_raw.shape}')
df_raw.head()

---
## 5. Limpieza y Preparación de Datos

Limpiamos el dataset: eliminamos valores nulos, seleccionamos columnas relevantes, y renombramos al español para facilitar la interpretación.

In [ ]:
df_clean, clean_path = clean(
    team_name=EQUIPO,
    output_dir=os.path.join('data', 'processed')
)

print(f'\nShape del dataset limpio: {df_clean.shape}')
print(f'Columnas: {df_clean.columns.tolist()}')
df_clean[['fecha', 'puntos', 'resultado', 'enfrentamiento']].head(10)

---
## 6. Análisis Exploratorio de Datos (EDA)

Exploramos la distribución de puntos anotados con estadísticas descriptivas y visualizaciones iniciales.

In [ ]:
puntos = df_clean['puntos'].values.astype(float)
n_partidos = len(puntos)

print(f'Total de partidos analizados: {n_partidos}')
print(f'\nEstadísticas descriptivas de PTS:')
print(df_clean['puntos'].describe())
print(f'\nModa: {df_clean["puntos"].mode().values[0]}')
print(f'Asimetría: {stats.skew(puntos):.3f}')
print(f'Curtosis: {stats.kurtosis(puntos):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.hist(puntos, bins=30, density=True, alpha=0.7, color='#2196F3',
        edgecolor='white', linewidth=0.5, label='Densidad observada')
ax.axvline(np.mean(puntos), color='#D32F2F', linestyle='--', linewidth=2,
           label=f'Media = {np.mean(puntos):.1f}')
ax.axvline(np.median(puntos), color='#4CAF50', linestyle='-.', linewidth=2,
           label=f'Mediana = {np.median(puntos):.1f}')
ax.set_xlabel('Puntos por partido')
ax.set_ylabel('Densidad')
ax.set_title(f'Distribución de Puntos — {NOMBRE_EQUIPO}')
ax.legend()

ax = axes[1]
ax.boxplot(puntos, vert=True, patch_artist=True,
           boxprops=dict(facecolor='#2196F3', alpha=0.6),
           medianprops=dict(color='#D32F2F', linewidth=2))
ax.set_ylabel('Puntos por partido')
ax.set_title(f'Diagrama de Caja — {NOMBRE_EQUIPO}')
ax.set_xticklabels([])

plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/eda_inicial.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Estadística Descriptiva Formal

Calculamos las estadísticas descriptivas formales usando nuestro módulo de análisis.

In [ ]:
stats_dict = estadistica_descriptiva(puntos)
lambda_est = stats_dict['lambda_estimado']

print(f'{"Métrica":<35} {"Valor":>10}')
print('-' * 47)
for k, v in stats_dict.items():
    if isinstance(v, float):
        print(f'{k:<35} {v:>10.3f}')
    else:
        print(f'{k:<35} {v:>10}')

print(f'\nLambda estimado (media muestral): {lambda_est:.3f}')

---
## 8. Verificación de la Propiedad Fundamental: $E[X] = Var(X) = \lambda$

Una condición necesaria (aunque no suficiente) para que los datos sigan una distribución de Poisson es que la media y la varianza sean aproximadamente iguales. La razón $\text{Var}(X)/E[X]$ debe ser cercana a 1.

- Si $\text{Var}/E[X] \approx 1$: posible Poisson (equidispersión)
- Si $\text{Var}/E[X] > 1$: sobredispersión (posible binomial negativa)
- Si $\text{Var}/E[X] < 1$: infradispersión (poco común en datos reales)

In [ ]:
propiedad = verificar_propiedad_poisson(stats_dict)

fig, ax = plt.subplots(figsize=(8, 5))
metricas = ['Media', 'Varianza']
valores = [propiedad['media'], propiedad['varianza']]
colores = ['#2196F3', '#FF9800']

bars = ax.bar(metricas, valores, color=colores, edgecolor='white', linewidth=1.5, width=0.4)
for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax.set_ylabel('Valor')
ax.set_title(f'Comparación Media vs Varianza — {NOMBRE_EQUIPO}\n'
             f'Razón Var/E[X] = {propiedad["ratio_varianza_media"]:.4f}')
ax.set_ylim(0, max(valores) * 1.2)

ax.text(0.5, 0.95, f'Diagnóstico: {propiedad["diagnostico"]}',
        transform=ax.transAxes, ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('plots/propiedad_media_varianza.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMedia (E[X]):           {propiedad["media"]:.2f}')
print(f'Varianza (Var[X]):       {propiedad["varianza"]:.2f}')
print(f'Razón Var/E[X]:         {propiedad["ratio_varianza_media"]:.4f}')
print(f'Diferencia relativa:    {propiedad["diferencia_relativa"]:.4f}')
print(f'\n{propiedad["diagnostico"]}')

---
## 9. Ajuste de la Distribución de Poisson

Estimamos $\hat{\lambda} = \bar{x}$ (método de máxima verosimilitud para la distribución de Poisson) y superponemos la PMF teórica sobre el histograma de datos reales.

In [ ]:
x_pmf, y_pmf = poisson_pmf_teorica(lambda_est)

fig, ax = plt.subplots(figsize=(12, 6))
min_val = int(min(puntos.min(), lambda_est - 4 * np.sqrt(lambda_est)))
max_val = int(max(puntos.max(), lambda_est + 4 * np.sqrt(lambda_est)))
bins = np.arange(min_val - 0.5, max_val + 1.5, 1)

ax.hist(puntos, bins=bins, density=True, alpha=0.65, color='#2196F3',
        edgecolor='white', linewidth=0.5, label='Datos reales (n={})'.format(n_partidos))
ax.plot(x_pmf, y_pmf, 'o-', color='#D32F2F', linewidth=2.5, markersize=5,
        label=f'Poisson teórica ($\\lambda$ = {lambda_est:.2f})')
ax.axvline(lambda_est, color='#D32F2F', linestyle='--', alpha=0.4,
           label=f'$\\lambda$ = {lambda_est:.2f}')

ax.set_xlabel('Puntos por partido (k)')
ax.set_ylabel('$P(X = k)$')
ax.set_title(f'Ajuste de Distribución de Poisson — {NOMBRE_EQUIPO}\n'
             f'$\\hat{{\\lambda}} = \\bar{{x}} = {lambda_est:.2f}$')
ax.legend(loc='upper right')
ax.set_xlim(min_val, max_val)

plt.tight_layout()
plt.savefig('plots/ajuste_poisson.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Prueba de Bondad de Ajuste $\chi^2$ (Chi-Cuadrada)

La prueba chi-cuadrada compara las frecuencias observadas $O_i$ con las frecuencias esperadas $E_i$ bajo la distribución de Poisson:

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i} \sim \chi^2_{(k - 2)}$$

Donde $k$ es el número de bins y restamos 2 grados de libertad porque estimamos $\lambda$ de los datos.

- **H₀:** Los datos siguen una distribución de Poisson($\lambda$)
- **H₁:** Los datos NO siguen una distribución de Poisson($\lambda$)

In [ ]:
chi2_result = prueba_chi_cuadrada(puntos, lambda_est, alpha=0.05)

if chi2_result['grados_libertad'] is not None and not np.isnan(chi2_result['grados_libertad']):
    print(f'Estadístico chi2:        {chi2_result["estadistico_chi2"]:.3f}')
    print(f'Grados de libertad:      {chi2_result["grados_libertad"]}')
    print(f'P-valor:                 {chi2_result["p_valor"]:.4f}')
    print(f'Chi2 crítico (alpha=0.05): {chi2_result["chi2_critico"]:.3f}')
    print(f'Nivel de significancia:  {chi2_result["nivel_significancia"]}')
    print(f'\nConclusión: {chi2_result["conclusion"]}')
    
    print(f'\nBins utilizados en la prueba:')
    for i, (label, obs, esp) in enumerate(zip(
        chi2_result['bins_labels'],
        chi2_result['observados'],
        chi2_result['esperados']
    )):
        print(f'  {label:>8}:  Obs={obs:5.1f}  Esp={esp:5.1f}  '
              f'Contribución={((obs-esp)**2/esp):.2f}')
else:
    print(chi2_result['conclusion'])

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

if chi2_result['grados_libertad'] is not None and not np.isnan(chi2_result['grados_libertad']):
    labels = chi2_result['bins_labels']
    obs = chi2_result['observados']
    esp = chi2_result['esperados']
    x = np.arange(len(labels))
    width = 0.35

    ax.bar(x - width/2, obs, width, label='Observadas',
           color='#2196F3', edgecolor='white', linewidth=0.5)
    ax.bar(x + width/2, esp, width, label='Esperadas (Poisson)',
           color='#FF9800', edgecolor='white', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_xlabel('Rango de puntos por partido')
    ax.set_ylabel('Frecuencia')
    ax.set_title(f'Prueba $\\chi^2$: Frecuencias Observadas vs Esperadas — {NOMBRE_EQUIPO}')
    ax.legend(loc='upper right')

    ax.text(0.98, 0.95,
            f'$\\chi^2$ = {chi2_result["estadistico_chi2"]:.2f}\n'
            f'gl = {chi2_result["grados_libertad"]}\n'
            f'p = {chi2_result["p_valor"]:.4f}\n'
            f'$\\chi^2_{{\\text{{crítico}}}}$ = {chi2_result["chi2_critico"]:.2f}',
            transform=ax.transAxes, fontsize=11, verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

    plt.tight_layout()
    plt.savefig('plots/barras_chi2.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No se puede generar gráfica: prueba chi2 no válida.')

---
## 11. Prueba de Kolmogorov-Smirnov (KS)

La prueba KS compara la función de distribución acumulada empírica (ECDF) de los datos con la CDF teórica de Poisson:

$$D_n = \sup_x |F_n(x) - F(x)|$$

Dado que estimamos $\lambda$ de los datos, el p-valor estándar es conservador. Para corregirlo, usamos una simulación Monte Carlo con 10,000 réplicas donde en cada iteración simulamos datos, estimamos $\lambda$ y recalculamos el estadístico KS.

In [ ]:
print('Ejecutando prueba KS con simulación Monte Carlo (10,000 iteraciones)...')
ks_result = prueba_ks(puntos, lambda_est, alpha=0.05, n_sim=10000)

print(f'\nEstadístico KS:          {ks_result["estadistico_ks"]:.4f}')
print(f'P-valor estándar:        {ks_result["p_valor_estandar"]:.4f}')
print(f'P-valor Monte Carlo:     {ks_result["p_valor_monte_carlo"]:.4f}')
print(f'Simulaciones realizadas: {ks_result["n_simulaciones"]}')
print(f'\nConclusión: {ks_result["conclusion"]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

p_sorted = np.sort(puntos)
n = len(puntos)
ecdf = np.arange(1, n + 1) / n
tcdf = poisson.cdf(p_sorted, mu=lambda_est)

ax = axes[0]
ax.step(p_sorted, ecdf, where='post', color='#2196F3', linewidth=2, label='ECDF (empírica)')
ax.step(p_sorted, tcdf, where='post', color='#D32F2F', linewidth=2, label=f'CDF Poisson($\\lambda$={lambda_est:.1f})')

max_diff_idx = np.argmax(np.abs(ecdf - tcdf))
ax.vlines(p_sorted[max_diff_idx], ecdf[max_diff_idx], tcdf[max_diff_idx],
          color='#FF9800', linewidth=2.5,
          label=f'$D_n$ = {ks_result["estadistico_ks"]:.4f}')

ax.set_xlabel('Puntos por partido')
ax.set_ylabel('Probabilidad acumulada')
ax.set_title(f'Prueba KS — {NOMBRE_EQUIPO}')
ax.legend(loc='lower right')

ax = axes[1]
theoretical_quantiles = poisson.ppf((np.arange(1, n + 1) - 0.5) / n, mu=lambda_est)
ax.scatter(theoretical_quantiles, p_sorted, alpha=0.4, color='#2196F3',
           edgecolors='white', linewidth=0.3, s=40)
ax.plot(theoretical_quantiles, theoretical_quantiles, '--', color='#D32F2F',
        linewidth=2, label='Referencia y = x')
ax.set_xlabel('Cuantiles teóricos Poisson')
ax.set_ylabel('Cuantiles observados')
ax.set_title(f'Q-Q Plot Poisson — {NOMBRE_EQUIPO}')
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('plots/prueba_ks.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Simulación Monte Carlo

Simulamos $n$ partidos usando una distribución de Poisson con $\lambda = \hat{\lambda}$. Comparamos visualmente la distribución empírica real con la distribución simulada.

In [ ]:
np.random.seed(42)
puntos_sim = simular_poisson(lambda_est, n_partidos, seed=42)

print(f'Simulación de {n_partidos} partidos con lambda = {lambda_est:.2f}')
print(f'\nComparación Real vs Simulado:')
print(f'  Media real:     {np.mean(puntos):.2f}')
print(f'  Media simulada: {np.mean(puntos_sim):.2f}')
print(f'  Var real:       {np.var(puntos, ddof=0):.2f}')
print(f'  Var simulada:   {np.var(puntos_sim, ddof=0):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

p_real = puntos.astype(int)
p_sim = puntos_sim.astype(int)

both = np.concatenate([p_real, p_sim])
min_all = both.min()
max_all = both.max()
bins = np.arange(min_all - 0.5, max_all + 1.5, 1)

ax = axes[0]
ax.hist(p_real, bins=bins, density=True, alpha=0.7, color='#2196F3',
        edgecolor='white', linewidth=0.5, label='Reales')
x_pmf, y_pmf = poisson_pmf_teorica(lambda_est, x_min=min_all, x_max=max_all)
ax.plot(x_pmf, y_pmf, 'o-', color='#D32F2F', linewidth=2, markersize=3,
        label=f'Poisson($\\lambda$={lambda_est:.1f})')
ax.set_title(f'Datos Reales (n={n_partidos})')
ax.set_xlabel('Puntos')
ax.set_ylabel('Densidad')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(p_sim, bins=bins, density=True, alpha=0.7, color='#FF9800',
        edgecolor='white', linewidth=0.5, label='Simulados')
ax.plot(x_pmf, y_pmf, 'o-', color='#D32F2F', linewidth=2, markersize=3,
        label=f'Poisson($\\lambda$={lambda_est:.1f})')
ax.set_title(f'Simulación Poisson (n={n_partidos})')
ax.set_xlabel('Puntos')
ax.set_ylabel('Densidad')
ax.legend(fontsize=9)

ax = axes[2]
ax.hist(p_real, bins=bins, density=True, alpha=0.5, color='#2196F3',
        edgecolor='white', linewidth=0.5, label='Reales')
ax.hist(p_sim, bins=bins, density=True, alpha=0.4, color='#FF9800',
        edgecolor='white', linewidth=0.5, label='Simulados')
ax.plot(x_pmf, y_pmf, 'o-', color='#D32F2F', linewidth=2, markersize=3,
        label=f'Poisson($\\lambda$={lambda_est:.1f})')
ax.set_title('Superposición Real vs Simulado')
ax.set_xlabel('Puntos')
ax.set_ylabel('Densidad')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plots/comparacion_monte_carlo.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 13. Serie Temporal: Estacionariedad de $\lambda$

Un proceso de Poisson homogéneo asume que la tasa $\lambda$ es constante en el tiempo. Graficamos los puntos anotados cronológicamente para verificar visualmente si hay tendencias o cambios estructurales.

In [ ]:
graficar_serie_temporal(df_clean, equipo=NOMBRE_EQUIPO, plots_dir='plots')

fig, ax = plt.subplots(figsize=(14, 5))

media_global = df_clean['puntos'].mean()
rolling_mean = df_clean['puntos'].rolling(window=20, center=True).mean()

ax.plot(df_clean['fecha'], df_clean['puntos'], 'o', markersize=2.5,
        color='#2196F3', alpha=0.5, label='Puntos por partido')
ax.plot(df_clean['fecha'], rolling_mean, '-', color='#D32F2F', linewidth=2.5,
        label='Media móvil (ventana=20)')
ax.axhline(media_global, color='#333333', linestyle='--', linewidth=1.5,
           label=f'Media global = {media_global:.1f}')

ax.set_xlabel('Fecha del partido')
ax.set_ylabel('Puntos anotados')
ax.set_title(f'Serie Temporal con Media Móvil — {NOMBRE_EQUIPO}')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('plots/serie_temporal_detalle.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 14. Media y Varianza por Temporada

Para evaluar la estabilidad del parámetro $\lambda$ a través de las temporadas.

In [ ]:
df_temp = df_clean.copy()
df_temp['temporada'] = df_temp['fecha'].apply(
    lambda d: f'{d.year}-{str(d.year+1)[-2:]}' if d.month >= 10
    else f'{d.year-1}-{str(d.year)[-2:]}'
)

agg = df_temp.groupby('temporada')['puntos'].agg(['mean', 'var', 'count'])
agg = agg[agg['count'] >= 10]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(agg))

ax.bar(x - 0.15, agg['mean'], 0.3, label='Media', color='#2196F3',
       edgecolor='white', linewidth=0.5)
ax.bar(x + 0.15, agg['var'], 0.3, label='Varianza', color='#FF9800',
       edgecolor='white', linewidth=0.5)
ax.axhline(df_clean['puntos'].mean(), color='#D32F2F', linestyle=':',
           linewidth=2, label=f'Media global = {df_clean["puntos"].mean():.1f}')

ax.set_xticks(x)
ax.set_xticklabels(agg.index, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Temporada')
ax.set_ylabel('Puntos')
ax.set_title(f'Media y Varianza por Temporada — {NOMBRE_EQUIPO}')
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('plots/media_varianza_temporada.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTabla de media y varianza por temporada:')
print(agg.to_string())

---
## 15. Resumen Final y Conclusiones

Recopilamos todos los resultados estadísticos en una tabla resumen y emitimos la conclusión final sobre si el modelo Poisson es adecuado.

In [ ]:
resumen = resumen_estadistico(stats_dict, propiedad, chi2_result, ks_result)
imprimir_resumen(resumen)

summary_data = {
    'Métrica': [
        'Partidos (n)', 'Media (lambda)', 'Varianza',
        'Desv. Estándar', 'Mínimo', 'Máximo', 'Mediana',
        'Asimetría', 'Curtosis', 'Razón Var/Media',
        'Chi2 estadístico', 'Chi2 p-valor', 'Chi2 g.l.',
        'KS estadístico', 'KS p-valor (MC)'
    ],
    'Valor': [
        stats_dict['n'], f"{stats_dict['media']:.2f}", f"{stats_dict['varianza']:.2f}",
        f"{stats_dict['desviacion_estandar']:.2f}", f"{stats_dict['minimo']:.0f}",
        f"{stats_dict['maximo']:.0f}", f"{stats_dict['mediana']:.0f}",
        f"{stats_dict['asimetria']:.3f}", f"{stats_dict['curtosis']:.3f}",
        f"{propiedad['ratio_varianza_media']:.4f}",
        f"{chi2_result['estadistico_chi2']:.3f}", f"{chi2_result['p_valor']:.4f}",
        f"{chi2_result['grados_libertad']}",
        f"{ks_result['estadistico_ks']:.4f}", f"{ks_result['p_valor_monte_carlo']:.4f}"
    ]
}

df_summary = pd.DataFrame(summary_data)
print('\nTabla resumen:')
print(df_summary.to_string(index=False))

df_summary.to_csv('data/processed/resumen_estadistico.csv', index=False)
print('\nResumen guardado en: data/processed/resumen_estadistico.csv')

---
## 16. Interpretación Matemática

### ¿Qué significa que los puntos sigan un proceso de Poisson?

Si $X \sim \text{Poisson}(\lambda)$, donde $X$ son los puntos anotados en un partido:

1. **$\lambda$ representa la tasa promedio de anotación.** Por ejemplo, $\lambda = 115$ significa que en promedio el equipo anota 115 puntos por partido.

2. **La probabilidad de anotar exactamente $k$ puntos** se calcula como:
   $$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

3. **La desviación estándar es $\sqrt{\lambda}$.** Para $\lambda = 115$, $\sigma \approx 10.7$.

4. **Intervalo de predicción (aproximadamente el 95% de los partidos):**
   $$[\lambda - 2\sqrt{\lambda},\; \lambda + 2\sqrt{\lambda}]$$

5. **Los "eventos" subyacentes** (canastas de 1, 2 o 3 puntos) ocurren con una tasa combinada que resulta en $\lambda$ puntos esperados.

### Limitaciones del modelo Poisson

- **Supuesto de independencia:** Los puntos anotados en un minuto deberían ser independientes de los anotados en otro minuto. En realidad, hay rachas y momentos de presión.
- **Tasa constante:** El modelo asume $\lambda$ constante todo el partido, pero el ritmo varía (últimos minutos, tiempo extra).
- **Sobredispersión:** Si $\text{Var}(X) > E[X]$, el modelo Poisson subestima la variabilidad real. En ese caso, una distribución binomial negativa es más apropiada.

### Extensiones sugeridas
1. **Poisson por cuarto:** Modelar puntos en intervalos de 12 minutos para validar la propiedad de incrementos estacionarios.
2. **Regresión Poisson:** Incluir covariables (rival, local/visitante, días de descanso).
3. **Proceso de Poisson no homogéneo:** Permitir que $\lambda(t)$ varíe dentro del partido (mayor intensidad al final).
4. **Comparar equipos:** Analizar si equipos defensivos ajustan mejor o peor que equipos ofensivos.
5. **Cadenas de Markov:** Modelar transiciones entre estados de anotación (rachas de puntos).

In [ ]:
print(f'\nIntervalo del 95% aproximado:')
low = lambda_est - 2 * np.sqrt(lambda_est)
high = lambda_est + 2 * np.sqrt(lambda_est)
print(f'  [{low:.1f}, {high:.1f}] puntos por partido')

print(f'\nProbabilidades de anotación según el modelo:')
print(f'  P(X = {int(lambda_est)}) = {poisson.pmf(int(lambda_est), mu=lambda_est):.4f}')
print(f'  P(X < {int(lambda_est - np.sqrt(lambda_est))}) = '
      f'{poisson.cdf(int(lambda_est - np.sqrt(lambda_est)), mu=lambda_est):.4f}')
print(f'  P(X > {int(lambda_est + np.sqrt(lambda_est))}) = '
      f'{1 - poisson.cdf(int(lambda_est + np.sqrt(lambda_est)), mu=lambda_est):.4f}')

---
## 17. Referencias

1. Ross, S. M. (2014). *Introduction to Probability Models*. Academic Press.
2. Taylor, H. M., & Karlin, S. (1998). *An Introduction to Stochastic Modeling*. Academic Press.
3. nba_api Documentation. (2024). https://github.com/swar/nba_api
4. NBA.com Official Statistics. https://www.nba.com/stats
5. Law, A. M. (2015). *Simulation Modeling and Analysis*. McGraw-Hill.
6. D'Agostino, R. B., & Stephens, M. A. (1986). *Goodness-of-Fit Techniques*. CRC Press.

---

*Proyecto desarrollado para la materia de Procesos Estocásticos.*